# Travailler avec des strings — `.str`

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from _data import load_accidents

df = load_accidents()

# Colonnes textuelles disponibles
df.select_dtypes("object").columns.tolist()

## 1. Le principe : l'accesseur `.str`

Pandas expose toutes les méthodes des strings Python via l'accesseur `.str`.
Chaque méthode s'applique **vectorialement** à toute la colonne — pas de boucle.

In [ ]:
# Aperçu de la colonne adr (adresse de l'accident)
df["adr"].dropna().head(10)

## 2. Recherche : `.str.contains()`, `.str.startswith()`, `.str.endswith()`

In [ ]:
# Accidents sur une rue (contient "RUE" ou "rue")
df[df["adr"].str.contains("RUE", na=False, case=False)].shape

In [ ]:
# case=False : insensible à la casse
# na=False : les NaN donnent False au lieu de NaN (évite les erreurs en aval)
df["adr"].str.contains("AUTOROUTE", na=False, case=False).sum()

In [ ]:
# .startswith() / .endswith() — plus simple que contains pour les préfixes/suffixes
df["dep"].str.startswith("7").sum()  # départements 70-79 + 2A/2B et autres

In [ ]:
# .contains() avec regex
# Accidents sur une autoroute (A1, A6, A7…) ou une nationale (N7, N10…)
df[df["adr"].str.contains(r"\b[AN]\d+\b", na=False, regex=True)].shape

## 3. Normalisation : `.str.lower()`, `.str.upper()`, `.str.strip()`, `.str.title()`

In [ ]:
# La colonne adr mélange majuscules, espaces en trop, tirets...
# Normalisation en une chaîne
adr_propre = (
    df["adr"]
    .str.strip()          # supprimer espaces en début/fin
    .str.upper()          # tout en majuscules pour uniformiser
    .str.replace(r"\s+", " ", regex=True)  # espaces multiples → un seul
)
adr_propre.dropna().head(5)

In [ ]:
# .str.title() : première lettre de chaque mot en majuscule
df["adr"].str.title().dropna().head(3)

## 4. Remplacement : `.str.replace()`

In [ ]:
# Remplacement simple
df["adr"].str.replace("AVENUE", "AV.", regex=False).dropna().head(5)

In [ ]:
# Avec regex — capturer et réorganiser
# Exemple : normaliser "RUE DE LA" → "R. DE LA"
df["adr"].str.replace(r"^RUE ", "R. ", regex=True).dropna().head(5)

In [ ]:
# Supprimer des caractères parasites
# .str.replace() chaîné pour plusieurs remplacements
(
    df["adr"]
    .str.replace(r"[^A-Z0-9\s]", "", regex=True)  # garder lettres, chiffres, espaces
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .dropna()
    .head(5)
)

## 5. Découpage : `.str.split()` et `.str.extract()`

In [ ]:
# .str.split() — découper sur un séparateur
df["hrmn"].str.split(":").head(5)  # ['07', '40']

In [ ]:
# expand=True : chaque élément devient une colonne
heure_minute = df["hrmn"].str.split(":", expand=True)
heure_minute.columns = ["heure", "minute"]
heure_minute = heure_minute.astype(int)
heure_minute.head(5)

In [ ]:
# Utilisation en pipeline — ajouter heure et minute au DataFrame
(
    df
    .assign(
        heure =lambda df_: df_["hrmn"].str[:2].astype(int),
        minute=lambda df_: df_["hrmn"].str[3:].astype(int),
    )
    [["hrmn", "heure", "minute"]]
    .head(5)
)

In [ ]:
# .str.extract() — extraire avec un groupe de capture regex
# Extraire le numéro de voie dans l'adresse (ex: "12 RUE DE LA PAIX" → "12")
df["adr"].str.extract(r"^(\d+)").rename(columns={0: "num_voie"}).dropna().head(5)

In [ ]:
# .str.extractall() — tous les matches (retourne un MultiIndex)
# Trouver tous les nombres dans l'adresse
df["adr"].head(10).str.extractall(r"(\d+)").rename(columns={0: "nombre"})

## 6. Informations : `.str.len()`, `.str.count()`, `.str.isdigit()`

In [ ]:
# Longueur des adresses
df["adr"].str.len().describe()

In [ ]:
# Nombre d'espaces dans l'adresse (proxy pour le nombre de mots)
df["adr"].str.count(r"\s+").value_counts().head(8)

In [ ]:
# Vérifications de type
sample = pd.Series(["75", "2A", "01", "974", "abc"])
print(".str.isdigit() :", sample.str.isdigit().tolist())
print(".str.isnumeric():", sample.str.isnumeric().tolist())
print(".str.isalpha() :", sample.str.isalpha().tolist())

## 7. `.str` pour les colonnes `dep` — nettoyage réel

In [ ]:
# Quelques valeurs de dep
df["dep"].value_counts().head(10)

In [ ]:
# Vérifier si des deps ont une longueur anormale (devrait être 2 ou 3 pour DOM)
df["dep"].str.len().value_counts()

In [ ]:
# Padding à 2 caractères — utile pour trier ou afficher
df["dep"].str.zfill(2).value_counts().head(5)  # '1' → '01'

## Récapitulatif

| Besoin | Méthode |
|---|---|
| Recherche simple | `.str.contains("x", na=False)` |
| Recherche regex | `.str.contains(r"pattern", regex=True, na=False)` |
| Préfixe / suffixe | `.str.startswith()` / `.str.endswith()` |
| Minuscule / majuscule | `.str.lower()` / `.str.upper()` / `.str.title()` |
| Supprimer espaces | `.str.strip()` / `.str.lstrip()` / `.str.rstrip()` |
| Remplacer | `.str.replace("old", "new", regex=False)` |
| Découper | `.str.split("sep", expand=True)` |
| Extraire un groupe regex | `.str.extract(r"(pattern)")` |
| Longueur | `.str.len()` |
| Compter les occurrences | `.str.count(r"pattern")` |
| Padding | `.str.zfill(n)` / `.str.pad(n, side="left")` |